# Nível 1 — Dados e primeira análise com LLM

**Parte A — Tratamento e regras (pandas)**
1. Carregar `dados/dados_nivel_1.json`
2. Limpar dados (documentar problemas encontrados e por que foram tratados assim)
3. Normalizar valores para BRL usando `taxa_cambio_usd_brl`
4. Agregações: volume total por cliente; contagem de operações por canal
5. Regra 1 — Fracionamento
6. Regra 2 — Valor atípico
7. Validação das regras (mostrar caso capturado e caso parecido não capturado)

**Parte B — Análise com LLM**
1. Escolher cliente sinalizado, montar prompt para parecer estruturado
2. Validar saída (nivel_risco, tipologia_suspeita, red_flags, justificativa)
3. Tratar resposta malformada
4. Registrar tokens e tempo de resposta
5. Comparar duas versões de prompt

> ⚠️ Lembrete: entregar este notebook com as células já executadas — saída sem resultados commitados não pode ser avaliada.

In [1]:
import json
import pandas as pd

with open("../dados/dados_nivel_1.json", encoding="utf-8") as f:
    dados = json.load(f)

taxa_cambio_usd_brl = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])
df.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


## Limpeza dos dados

Problemas de qualidade encontrados na base e o tratamento aplicado:

1. **Registro duplicado.** A operação `OP-0007` (cliente `CLI-A-3`) aparece duas vezes, com todos os campos idênticos. Tratamento: removida a duplicata por `id` (`drop_duplicates`), mantendo uma única ocorrência. Contar a mesma operação duas vezes infla o volume do cliente e pode disparar falsos positivos nas regras (em especial a Regra 1, que soma valores por data).
2. **Data ausente.** A operação `OP-0017` (cliente `CLI-A-5`) tem `data = None`, com a observação *"data nao capturada pelo sistema"* — indício de falha de captura do sistema legado, não de um evento sem data real. Tratamento: a operação é **mantida** nas agregações que não dependem de data (volume total, contagem por canal, Regra 2), mas é **excluída** do agrupamento por data da Regra 1 (fracionamento), já que sem data não há como saber se ela ocorreu junto de outras operações no mesmo dia. Marcada com a flag `data_ausente` para rastreabilidade.
3. **Moeda mista.** A operação `OP-0013` está em USD, enquanto o restante da base está em BRL. Tratamento: convertida para BRL usando a `taxa_cambio_usd_brl` fixa do próprio arquivo, gerando a coluna `valor_brl`. Todas as agregações e regras a seguir usam `valor_brl`, nunca o `valor` bruto — misturar moedas sem converter distorceria qualquer soma ou comparação.

Nenhum outro problema estrutural foi encontrado (sem valores negativos ou nulos em `valor`, sem `cliente_id`/`canal`/`moeda` fora do domínio esperado).

In [2]:
# Diagnóstico de qualidade dos dados
print("Linhas totais:", len(df))
print("IDs duplicados:", df["id"].duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Moedas distintas:", df["moeda"].unique().tolist())
print("Valores nulos por coluna:")
print(df.isna().sum())

Linhas totais: 20
IDs duplicados: 1
Datas ausentes: 1
Moedas distintas: ['BRL', 'USD']
Valores nulos por coluna:
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64


In [3]:
# Remoção de duplicata exata (mesmo id) e marcação de data ausente
df_clean = df.drop_duplicates(subset=["id"]).copy()
df_clean["data_ausente"] = df_clean["data"].isna()

print(f"Linhas antes: {len(df)} -> depois de remover duplicata: {len(df_clean)}")
df_clean[df_clean["data_ausente"]]

Linhas antes: 20 -> depois de remover duplicata: 19


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema,True


In [4]:
# Normalização para BRL usando a taxa fixa do arquivo
df_clean["valor_brl"] = df_clean.apply(
    lambda r: r["valor"] * taxa_cambio_usd_brl if r["moeda"] == "USD" else r["valor"],
    axis=1,
)
df_clean[["id", "cliente_id", "moeda", "valor", "valor_brl"]][df_clean["moeda"] == "USD"]

,id,cliente_id,moeda,valor,valor_brl
13,OP-0013,CLI-A-4,USD,12000,64800.0


In [5]:
# Agregações
volume_por_cliente = (
    df_clean.groupby("cliente_id")["valor_brl"].sum().sort_values(ascending=False)
)
print("Volume total transacionado por cliente (BRL):")
display(volume_por_cliente)

contagem_por_canal = df_clean["canal"].value_counts()
print("\nQuantidade de operações por canal:")
display(contagem_por_canal)

Volume total transacionado por cliente (BRL):


cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: valor_brl, dtype: float64


Quantidade de operações por canal:


canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

## Regra 1 — Fracionamento

Sinaliza o cliente que, em uma mesma data, realizou 3 ou mais operações cuja soma
ultrapassa R\$ 50.000,00, sem que nenhuma operação isolada atinja R\$ 20.000,00.

Só entram no agrupamento por data as operações com `data` conhecida (ver decisão
de limpeza acima sobre `data_ausente`).

In [6]:
LIMITE_SOMA_FRACIONAMENTO = 50_000
LIMITE_OPERACAO_ISOLADA = 20_000
MIN_OPERACOES_FRACIONAMENTO = 3

com_data = df_clean[~df_clean["data_ausente"]]

por_dia = (
    com_data.groupby(["cliente_id", "data"])["valor_brl"]
    .agg(qtd_operacoes="count", soma_valor="sum", maior_operacao="max")
    .reset_index()
)

candidatos_fracionamento = por_dia[
    (por_dia["qtd_operacoes"] >= MIN_OPERACOES_FRACIONAMENTO)
    & (por_dia["soma_valor"] > LIMITE_SOMA_FRACIONAMENTO)
    & (por_dia["maior_operacao"] < LIMITE_OPERACAO_ISOLADA)
]
candidatos_fracionamento

,cliente_id,data,qtd_operacoes,soma_valor,maior_operacao
0,CLI-A-1,2026-03-09,3,54200.0,18800.0


In [7]:
# Flag por operação: True se a operação pertence a um grupo (cliente, data) sinalizado
chaves_fracionamento = set(
    zip(candidatos_fracionamento["cliente_id"], candidatos_fracionamento["data"])
)
df_clean["flag_fracionamento"] = df_clean.apply(
    lambda r: (r["cliente_id"], r["data"]) in chaves_fracionamento, axis=1
)

clientes_flag_fracionamento = sorted(candidatos_fracionamento["cliente_id"].unique().tolist())
print("Clientes sinalizados pela Regra 1 (fracionamento):", clientes_flag_fracionamento)
df_clean[df_clean["flag_fracionamento"]]

Clientes sinalizados pela Regra 1 (fracionamento): ['CLI-A-1']


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl,flag_fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False,18800.0,True


## Regra 2 — Valor atípico

Sinaliza a operação cujo `valor_brl` seja superior a 5x a mediana dos valores
daquele cliente. Aplicada apenas a clientes com 4 ou mais operações.

In [8]:
MULTIPLICADOR_ATIPICO = 5
MIN_OPERACOES_ATIPICO = 4

qtd_operacoes_cliente = df_clean.groupby("cliente_id")["valor_brl"].transform("count")
mediana_cliente = df_clean.groupby("cliente_id")["valor_brl"].transform("median")

df_clean["flag_valor_atipico"] = (qtd_operacoes_cliente >= MIN_OPERACOES_ATIPICO) & (
    df_clean["valor_brl"] > MULTIPLICADOR_ATIPICO * mediana_cliente
)

clientes_flag_atipico = sorted(
    df_clean.loc[df_clean["flag_valor_atipico"], "cliente_id"].unique().tolist()
)
print("Clientes com ao menos uma operação atípica (Regra 2):", clientes_flag_atipico)
df_clean[df_clean["flag_valor_atipico"]]

Clientes com ao menos uma operação atípica (Regra 2): ['CLI-A-4']


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl,flag_fracionamento,flag_valor_atipico
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional,False,64800.0,False,True


## Validação das regras

Foco na Regra 1: mostrar um caso que deveria ser capturado e um caso parecido
que não deveria.

- `CLI-A-1` fez 3 operações em `2026-03-09` somando R\$ 54.200, nenhuma isolada
  chegando a R\$ 20.000 → **deve** ser sinalizado.
- `CLI-A-3` fez 3 operações em `2026-03-05` somando R\$ 48.500 (perfil muito
  parecido: mesma quantidade de operações, mesmo canal, valores na mesma faixa)
  mas a soma fica **abaixo** do limite de R\$ 50.000 → **não deve** ser sinalizado.

In [9]:
caso_positivo = df_clean[
    (df_clean["cliente_id"] == "CLI-A-1") & (df_clean["data"] == "2026-03-09")
][["cliente_id", "data", "valor_brl", "flag_fracionamento"]]

caso_negativo = df_clean[
    (df_clean["cliente_id"] == "CLI-A-3") & (df_clean["data"] == "2026-03-05")
][["cliente_id", "data", "valor_brl", "flag_fracionamento"]]

print("Caso que DEVE ser capturado (CLI-A-1, soma R$ 54.200 > R$ 50.000):")
display(caso_positivo)
print(f"Soma: {caso_positivo['valor_brl'].sum():.2f} | Capturado: {caso_positivo['flag_fracionamento'].any()}")

print("\nCaso PARECIDO que NÃO deve ser capturado (CLI-A-3, soma R$ 48.500 < R$ 50.000):")
display(caso_negativo)
print(f"Soma: {caso_negativo['valor_brl'].sum():.2f} | Capturado: {caso_negativo['flag_fracionamento'].any()}")

assert caso_positivo["flag_fracionamento"].all(), "Regra 1 deveria capturar CLI-A-1"
assert not caso_negativo["flag_fracionamento"].any(), "Regra 1 nao deveria capturar CLI-A-3"
print("\nValidação OK: a Regra 1 captura o caso que deveria e não captura o caso parecido que não se enquadra.")

Caso que DEVE ser capturado (CLI-A-1, soma R$ 54.200 > R$ 50.000):


,cliente_id,data,valor_brl,flag_fracionamento
0,CLI-A-1,2026-03-09,18100.0,True
1,CLI-A-1,2026-03-09,17300.0,True
2,CLI-A-1,2026-03-09,18800.0,True


Soma: 54200.00 | Capturado: True

Caso PARECIDO que NÃO deve ser capturado (CLI-A-3, soma R$ 48.500 < R$ 50.000):


,cliente_id,data,valor_brl,flag_fracionamento
6,CLI-A-3,2026-03-05,17200.0,False
7,CLI-A-3,2026-03-05,15200.0,False
8,CLI-A-3,2026-03-05,16100.0,False


Soma: 48500.00 | Capturado: False

Validação OK: a Regra 1 captura o caso que deveria e não captura o caso parecido que não se enquadra.
